In [0]:
# Phase 5 – Databricks + Olist: End-to-End Data Engineering Pipeline
# STEP 0: Verify Databricks Setup
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
spark = SparkSession.builder.appName("phase5").getOrCreate()
print("Spark Version:", spark.version)
print("Databricks Runtime is ready!")

Spark Version: 4.1.0
Databricks Runtime is ready!


In [0]:
# STEP 1: Upload & Load All CSV Datasets 
df_orders = spark.read.csv('/Volumes/workspace/default/olist/olist_orders_dataset.csv', header = True, inferSchema=True)
df_customers    = spark.read.csv('/Volumes/workspace/default/olist/olist_customers_dataset.csv', header = True, inferSchema=True)
df_order_items  = spark.read.csv('/Volumes/workspace/default/olist/olist_order_items_dataset.csv', header= True, inferSchema=True)
df_payments     = spark.read.csv('/Volumes/workspace/default/olist/olist_order_payments_dataset.csv', header= True, inferSchema=True)
df_reviews      = spark.read.csv('/Volumes/workspace/default/olist/olist_order_reviews_dataset.csv', header= True, inferSchema=True)
df_products     = spark.read.csv('/Volumes/workspace/default/olist/olist_products_dataset.csv', header= True, inferSchema=True)
df_sellers      = spark.read.csv('/Volumes/workspace/default/olist/olist_sellers_dataset.csv', header= True, inferSchema=True)
df_category     = spark.read.csv('/Volumes/workspace/default/olist/product_category_name_translation.csv', header= True, inferSchema=True)
df_geo          = spark.read.csv('/Volumes/workspace/default/olist/olist_geolocation_dataset.csv', header= True, inferSchema=True)

In [0]:
print("All datasets loaded!")
print(f"  orders:       {df_orders.count():,} rows")
print(f"  customers:    {df_customers.count():,} rows")
print(f"  order_items:  {df_order_items.count():,} rows")
print(f"  payments:     {df_payments.count():,} rows")
print(f"  reviews:      {df_reviews.count():,} rows")
print(f"  products:     {df_products.count():,} rows")
print(f"  sellers:      {df_sellers.count():,} rows")
print(f"  category:     {df_category.count():,} rows")
print(f"  geolocation:  {df_geo.count():,} rows")

All datasets loaded!
  orders:       99,441 rows
  customers:    99,441 rows
  order_items:  112,650 rows
  payments:     103,886 rows
  reviews:      104,162 rows
  products:     32,951 rows
  sellers:      3,095 rows
  category:     71 rows
  geolocation:  1,000,163 rows


# DATA CLEANING: Null handling + Type Casting

In [0]:
# Cast timestamps
df_orders = df_orders.withColumn(
    "order_purchase_timestamp", to_timestamp("order_purchase_timestamp")
).withColumn(
    "order_delivered_customer_date", to_timestamp("order_delivered_customer_date")
).withColumn(
    "order_estimated_delivery_date", to_timestamp("order_estimated_delivery_date")
)

In [0]:
# Drop rows with nulls in critical columns
df_orders_clean     = df_orders.dropna(subset=["order_id", "customer_id", "order_purchase_timestamp"])
df_order_items_clean = df_order_items.dropna(subset=["order_id", "product_id", "price"])
df_payments_clean   = df_payments.dropna(subset=["order_id", "payment_value"])
df_customers_clean  = df_customers.dropna(subset=["customer_id", "customer_city"])

In [0]:
print("Data cleaning complete")
print(f"  orders after clean: {df_orders_clean.count():,} rows")
print(f"  order_items after clean: {df_order_items_clean.count():,} rows")
print(f"  payments after clean: {df_payments_clean.count():,} rows")

Data cleaning complete
  orders after clean: 99,441 rows
  order_items after clean: 112,650 rows
  payments after clean: 103,886 rows


In [0]:
# Referential Integrity Check: All order_items should have a valid order
orphan_items = df_order_items_clean.join(
    df_orders_clean.select("order_id"),
    on="order_id",
    how="left_anti"
)
print(f"Orphan order_items (no matching order): {orphan_items.count()}")

Orphan order_items (no matching order): 0


In [0]:
# All orders should have a matching customer
orphan_orders = df_orders_clean.join(
    df_customers_clean.select("customer_id"),
    on="customer_id",
    how="left_anti"
)
print(f"Orphan orders (no matching customer): {orphan_orders.count()}")
print("Referential integrity validated")

Orphan orders (no matching customer): 0
Referential integrity validated


In [0]:
# TASK 1: Top 3 Customers per City (Window Function – RANK)
from pyspark.sql.window import Window
from pyspark.sql.functions import col, round, sum as _sum, dense_rank

# Total spend per customer via payments
df_customer_spend = (
    df_payments_clean
    .join(df_orders_clean.select("order_id", "customer_id"), on="order_id", how="inner")
    .groupBy("customer_id")
    .agg(round(_sum("payment_value"), 2).alias("total_spend"))
)

# Join with customer city info
df_customer_city_spend = (
    df_customer_spend
    .join(df_customers_clean.select("customer_id", "customer_city"), on="customer_id", how="inner")
)

# Window: rank within each city by total_spend descending
city_window = Window.partitionBy("customer_city").orderBy(col("total_spend").desc())

df_task1 = (
    df_customer_city_spend
    .withColumn("rank", dense_rank().over(city_window))
    .filter(col("rank") <= 3)
    .select(
        col("customer_city").alias("city"),
        "customer_id",
        "total_spend",
        "rank"
    )
    .orderBy("city", "rank")
)

print("✅ Task 1: Top 3 Customers per City")
df_task1.show(5)

✅ Task 1: Top 3 Customers per City
+-------------------+--------------------+-----------+----+
|               city|         customer_id|total_spend|rank|
+-------------------+--------------------+-----------+----+
|abadia dos dourados|9e01f714a2b3b8962...|     219.63|   1|
|abadia dos dourados|a23e3f9a2b656b23b...|     135.59|   2|
|abadia dos dourados|f11eb8f0b8b87510a...|      58.28|   3|
|          abadiania|576d71ddb21b21763...|    1025.52|   1|
|             abaete|d47c8bb51df6f716e...|     466.89|   1|
+-------------------+--------------------+-----------+----+
only showing top 5 rows


In [0]:
# TASK 2: Running Total of Daily Sales (Window Function – cumSum)
from pyspark.sql.functions import to_date, dayofyear

# Daily sales from payments joined with orders (to get purchase date)
df_daily_sales = (
    df_payments_clean
    .join(df_orders_clean.select("order_id", "order_purchase_timestamp"), on="order_id", how="inner")
    .withColumn("date", to_date("order_purchase_timestamp"))
    .groupBy("date")
    .agg(round(_sum("payment_value"), 2).alias("daily_sales"))
    .orderBy("date")
)

# Cumulative running total using Window function
running_window = Window.orderBy("date").rowsBetween(Window.unboundedPreceding, Window.currentRow)

df_task2 = (
    df_daily_sales
    .withColumn("running_total", round(_sum("daily_sales").over(running_window), 2))
    .select("date", "daily_sales", "running_total")
)

print("✅ Task 2: Running Total of Daily Sales")
df_task2.show(5)

✅ Task 2: Running Total of Daily Sales


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+----------+-----------+-------------+
|      date|daily_sales|running_total|
+----------+-----------+-------------+
|2016-09-04|     136.23|       136.23|
|2016-09-05|      75.06|       211.29|
|2016-09-13|      40.95|       252.24|
|2016-10-02|     109.34|       361.58|
|2016-10-03|     595.14|       956.72|
+----------+-----------+-------------+
only showing top 5 rows


In [0]:
# TASK 3: Top Products per Category (DENSE_RANK)
# Aggregate sales per product
df_product_sales = (
    df_order_items_clean
    .groupBy("product_id")
    .agg(round(_sum("price"), 2).alias("total_sales"))
)

# Join with product table to get category name
df_product_category = (
    df_product_sales
    .join(df_products.select("product_id", "product_category_name"), on="product_id", how="inner")
    .join(df_category, on="product_category_name", how="left")
    .withColumn(
        "category",
        when(col("product_category_name_english").isNotNull(), col("product_category_name_english"))
        .otherwise(col("product_category_name"))
    )
)

# Rank within each category by total_sales
category_window = Window.partitionBy("category").orderBy(col("total_sales").desc())

df_task3 = (
    df_product_category
    .withColumn("rank", dense_rank().over(category_window))
    .filter(col("rank") <= 3)
    .select("category", "product_id", "total_sales", "rank")
    .orderBy("category", "rank")
)

print("✅ Task 3: Top Products per Category (DENSE_RANK)")
df_task3.show(5)

✅ Task 3: Top Products per Category (DENSE_RANK)
+--------------------+--------------------+-----------+----+
|            category|          product_id|total_sales|rank|
+--------------------+--------------------+-----------+----+
|                NULL|5a848e4ab52fd5445...|   24229.03|   1|
|                NULL|eed5cbd74fac3bd79...|     9945.0|   2|
|                NULL|b1d207586fca400a2...|     7152.0|   3|
|agro_industry_and...|11250b0d4b709fee9...|     9111.0|   1|
|agro_industry_and...|423a6644f0aa529e8...|     8043.0|   2|
+--------------------+--------------------+-----------+----+
only showing top 5 rows


In [0]:
# TASK 4: Customer Lifetime Value (CLV)

df_task4 = (
    df_payments_clean
    .join(df_orders_clean.select("order_id", "customer_id"), on="order_id", how="inner")
    .groupBy("customer_id")
    .agg(round(_sum("payment_value"), 2).alias("total_spend"))
    .orderBy(col("total_spend").desc())
)

print("✅ Task 4: Customer Lifetime Value")
print(f"  Total unique customers with spend: {df_task4.count():,}")
df_task4.show(5)

✅ Task 4: Customer Lifetime Value
  Total unique customers with spend: 99,440
+--------------------+-----------+
|         customer_id|total_spend|
+--------------------+-----------+
|1617b1357756262bf...|   13664.08|
|ec5b2ba62e5743423...|    7274.88|
|c6e2731c5b391845f...|    6929.31|
|f48d464a0baaea338...|    6922.21|
|3fd6777bbce08a352...|    6726.66|
+--------------------+-----------+
only showing top 5 rows


In [0]:
# TASK 5: Customer Segmentation (Gold / Silver / Bronze)

df_segmented = (
    df_task4
    .withColumn(
        "segment",
        when(col("total_spend") > 10000, "Gold")
        .when((col("total_spend") >= 5000) & (col("total_spend") <= 10000), "Silver")
        .otherwise("Bronze")
    )
)

# Per-customer output
df_task5_customers = df_segmented.select("customer_id", "total_spend", "segment")

# Summary: count per segment
df_task5_summary = (
    df_segmented
    .groupBy("segment")
    .agg(count("customer_id").alias("customer_count"))
    .orderBy("segment")
)

print("✅ Task 5: Customer Segmentation")
df_task5_customers.show(5)
print("\nSegment Summary:")
df_task5_summary.show(5)


✅ Task 5: Customer Segmentation
+--------------------+-----------+-------+
|         customer_id|total_spend|segment|
+--------------------+-----------+-------+
|1617b1357756262bf...|   13664.08|   Gold|
|ec5b2ba62e5743423...|    7274.88| Silver|
|c6e2731c5b391845f...|    6929.31| Silver|
|f48d464a0baaea338...|    6922.21| Silver|
|3fd6777bbce08a352...|    6726.66| Silver|
+--------------------+-----------+-------+
only showing top 5 rows

Segment Summary:
+-------+--------------+
|segment|customer_count|
+-------+--------------+
| Bronze|         99434|
|   Gold|             1|
| Silver|             5|
+-------+--------------+



In [0]:
# TASK 6: Final Reporting Table
# Combine: customer_id, city, total_spend, segment, total_orders

# Total orders per customer
df_order_counts = (
    df_orders_clean
    .groupBy("customer_id")
    .agg(count("order_id").alias("total_orders"))
)

# Combine everything
df_task6 = (
    df_segmented                                      # customer_id, total_spend, segment
    .join(
        df_customers_clean.select("customer_id", col("customer_city").alias("city")),
        on="customer_id", how="inner"
    )
    .join(df_order_counts, on="customer_id", how="left")
    .select("customer_id", "city", "total_spend", "segment", "total_orders")
    .orderBy(col("total_spend").desc())
)

print("✅ Task 6: Final Reporting Table")
print(f"  Total rows: {df_task6.count():,}")
df_task6.show(5)

✅ Task 6: Final Reporting Table
  Total rows: 99,440
+--------------------+--------------+-----------+-------+------------+
|         customer_id|          city|total_spend|segment|total_orders|
+--------------------+--------------+-----------+-------+------------+
|1617b1357756262bf...|rio de janeiro|   13664.08|   Gold|           1|
|ec5b2ba62e5743423...|    vila velha|    7274.88| Silver|           1|
|c6e2731c5b391845f...|  campo grande|    6929.31| Silver|           1|
|f48d464a0baaea338...|       vitoria|    6922.21| Silver|           1|
|3fd6777bbce08a352...|       marilia|    6726.66| Silver|           1|
+--------------------+--------------+-----------+-------+------------+
only showing top 5 rows
